In [1]:
# Implementation of a KAN for Brain Voxel Classification (1-vs-Rest)
# This notebook implements a Kolmogorov-Arnold Network (KAN) for 
# brain voxel classification using a 1-vs-rest approach.

# Cell 1: Initialize the environment and import libraries
import torch
from kan import *
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import pandas as pd
import random
import glob
from tqdm import tqdm

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Create folders for saving results
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('video_img', exist_ok=True)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

Using GPU: NVIDIA RTX A6000


In [2]:
# Cell 2: Hyperparameters Configuration

class Config:
    def __init__(self):
        # Data parameters
        self.feature_dim = 341  # Number of input features
        self.num_classes = 2    # Binary classification (1-vs-rest)
        self.negative_ratio = 10  # Ratio of negative to positive samples (1:k)
        self.val_negative_ratio = 5  # Validation set negative to positive ratio
        self.test_size = 0.2    # Fraction of data to use for testing
        self.random_state = 42  # Random seed for data splitting
        self.normalize_features = True  # Whether to standardize features
        
        # KAN model parameters
        self.network_width = [self.feature_dim, 256, 128, 64, self.num_classes]  # Network architecture
        self.grid_size = 20     # Fixed grid size (no grid expansion to avoid oscillation)
        self.k_value = 3        # Number of basis functions per dimension
        
        # Training parameters
        self.optimizer = "Adam"  # Optimizer type (Adam, SGD, etc.)
        self.learning_rate = 0.005  # Learning rate for optimizer
        self.weight_decay = 0.0001  # L2 regularization coefficient
        self.lambda_reg = 0.01    # Weight regularization coefficient
        self.lambda_entropy = 5.0  # Entropy regularization coefficient
        self.batch_size = 64      # Batch size for training
        self.train_steps = 100    # Number of training steps
        self.early_stopping = True  # Whether to use early stopping
        self.patience = 10        # Early stopping patience
        self.min_delta = 0.001    # Minimum improvement for early stopping

        # Class weighting for imbalanced data
        self.class_weights = torch.tensor([1.0, self.negative_ratio * 0.5], dtype=torch.float32)  # [pos_weight, neg_weight]
        
        # Pruning and fine-tuning
        self.enable_pruning = True  # Whether to prune the model after training
        self.fine_tune_steps = 50   # Number of fine-tuning steps after pruning
        
        # Evaluation metrics
        self.prediction_threshold = 0.5  # Threshold for binary predictions
        
        # Visualization parameters
        self.save_figures = True  # Whether to save figures
        self.img_folder = 'video_img'  # Folder to save training visualization images
        self.plot_frequency = 10   # How often to plot the model during training
        self.video_fps = 10        # Frames per second for training video
        
        # Symbolic regression parameters
        self.enable_symbolic = True  # Whether to extract symbolic expressions
        self.symbolic_library = ['x', 'x^2', 'exp', 'log', 'sqrt', 'sin', 'tanh', 'abs']  # Function library

    def display(self):
        """Display the current configuration"""
        print("=== KAN Brain Classification Configuration ===")
        print(f"Network Structure: {self.network_width}")
        print(f"Grid Size: {self.grid_size}, K Value: {self.k_value}")
        print(f"Negative to Positive Ratio: {self.negative_ratio}:1")
        print(f"Training Steps: {self.train_steps}, Batch Size: {self.batch_size}")
        print(f"Regularization: λ_reg={self.lambda_reg}, λ_entropy={self.lambda_entropy}")
        print(f"Learning Rate: {self.learning_rate}, Weight Decay: {self.weight_decay}")
        print(f"Early Stopping: {self.early_stopping} (patience={self.patience}, min_delta={self.min_delta})")
        print(f"Class Weights: {self.class_weights}")
        print("===============================================")

# Create a configuration object
config = Config()
config.display()

=== KAN Brain Classification Configuration ===
Network Structure: [341, 256, 128, 64, 2]
Grid Size: 20, K Value: 3
Negative to Positive Ratio: 10:1
Training Steps: 100, Batch Size: 64
Regularization: λ_reg=0.01, λ_entropy=5.0
Learning Rate: 0.005, Weight Decay: 0.0001
Early Stopping: True (patience=10, min_delta=0.001)
Class Weights: tensor([1., 5.])


In [5]:
# Cell 3: Data Loading and Processing Functions

def load_label_index(index_file):
    """Load label index file containing voxel counts for each label"""
    label_info = {}
    with open(index_file, 'r') as f:
        # Skip header
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                label_id = int(parts[0])
                voxel_count = int(parts[1])
                filename = parts[2] if parts[2] else None
                label_info[label_id] = {'count': voxel_count, 'filename': filename}
    return label_info

def get_label_file_path(label_id, is_validation=False, base_dir=None):
    """Get file path for a specific label"""
    if base_dir is None:
        if is_validation:
            base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_set_by_label'
        else:
            base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/train_set_by_label'
    
    pattern = os.path.join(base_dir, f"label_{label_id}_count_*_voxels.npy")
    matches = glob.glob(pattern)
    return matches[0] if matches else None

def create_1vs_rest_dataset(target_label_id, config, verbose=True):
    """
    Create a 1-vs-rest dataset for binary classification of a specific label
    
    Args:
        target_label_id: ID of the target label (positive class)
        config: Configuration object with hyperparameters
        verbose: Whether to print progress information
    
    Returns:
        dataset: Dictionary containing training and testing data
    """
    if verbose:
        print(f"Creating 1-vs-rest dataset for label {target_label_id}...")
    
    # 1. Load the training label index file to get information about all labels
    train_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/train_set_by_label'
    val_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_set_by_label'
    
    train_index_file = os.path.join(train_label_dir, "label_index.txt")
    val_index_file = os.path.join(val_label_dir, "val_label_index.txt")
    
    if not os.path.exists(train_index_file):
        raise FileNotFoundError(f"Training label index file not found: {train_index_file}")
    
    # Load label indices
    train_label_info = load_label_index(train_index_file)
    
    # Find valid labels (with voxel data)
    valid_labels = [label_id for label_id, info in train_label_info.items() 
                   if info['count'] > 0]
    
    if verbose:
        print(f"  Found {len(valid_labels)} valid labels with training data")
    
    # 2. Load the target label data (positive samples)
    target_file = get_label_file_path(target_label_id)
    if not target_file:
        raise ValueError(f"Label {target_label_id} has no corresponding data file")
    
    positive_samples = np.load(target_file)
    num_positive = len(positive_samples)
    
    if verbose:
        print(f"  Loaded {num_positive} positive samples from {target_file}")
    
    # 3. Calculate how many negative samples we need based on the ratio
    num_negative = num_positive * config.negative_ratio
    
    # Get a list of all other valid label IDs except the target
    other_labels = [l for l in valid_labels if l != target_label_id]
    
    # Randomly select which labels to use as negatives
    random.shuffle(other_labels)
    
    negative_samples = []
    negative_count = 0
    
    # For each selected label, load its data and add to negative samples
    for other_label_id in other_labels:
        if negative_count >= num_negative:
            break
            
        other_file = get_label_file_path(other_label_id)
        if not other_file:
            continue
            
        try:
            # Load data for this label
            label_samples = np.load(other_file)
            
            # Calculate how many samples to take from this label based on distribution
            # Here we distribute proportionally based on label size to avoid bias
            label_size = train_label_info[other_label_id]['count']
            total_other_size = sum(train_label_info[l]['count'] for l in other_labels)
            proportion = label_size / total_other_size if total_other_size > 0 else 0
            
            samples_needed = min(len(label_samples), max(10, int(num_negative * proportion)))
            
            # Randomly sample if we have more than needed
            if samples_needed < len(label_samples):
                indices = np.random.choice(len(label_samples), samples_needed, replace=False)
                sampled = label_samples[indices]
            else:
                sampled = label_samples
                
            negative_samples.append(sampled)
            negative_count += len(sampled)
            
            if verbose and len(negative_samples) <= 5:  # Only show first few labels for brevity
                print(f"  Added {len(sampled)} negative samples from label {other_label_id}")
                
        except Exception as e:
            print(f"  Error loading data for label {other_label_id}: {str(e)}")
    
    # Combine all negative samples
    if negative_samples:
        negative_samples = np.vstack(negative_samples)
        
        # If we have more than needed, randomly sample
        if len(negative_samples) > num_negative:
            indices = np.random.choice(len(negative_samples), int(num_negative), replace=False)
            negative_samples = negative_samples[indices]
    else:
        # If no negative samples were loaded, create synthetic ones
        if verbose:
            print("  Warning: No negative samples loaded. Creating synthetic negative samples.")
        negative_samples = np.random.randn(int(num_negative), config.feature_dim)
    
    if verbose:
        print(f"  Final negative sample count: {len(negative_samples)} from {len(other_labels)} labels")
    
    # 4. Create feature data X and labels y
    X = np.vstack([positive_samples, negative_samples])
    
    # Create labels: 1 for positive class, 0 for negative class
    y_positive = np.ones((num_positive, 1))
    y_negative = np.zeros((len(negative_samples), 1))
    y = np.vstack([y_positive, y_negative])
    
    # Convert to integer class labels for CrossEntropyLoss
    y_class = y.flatten().astype(np.int64)
    
    # 5. Shuffle the data
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)
    X = X[indices]
    y_class = y_class[indices]
    
    # 6. Normalize features if requested
    if config.normalize_features:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    # 7. Split into training and testing sets
    test_size = int(X.shape[0] * config.test_size)
    X_train, X_test = X[:-test_size], X[-test_size:]
    y_train, y_test = y_class[:-test_size], y_class[-test_size:]
    
    if verbose:
        print(f"  Total dataset size: {X.shape[0]} samples")
        print(f"  Training set: {X_train.shape[0]} samples")
        print(f"  Testing set: {X_test.shape[0]} samples")
        print(f"  Feature dimension: {X.shape[1]}")
        
        # Calculate class distribution
        train_pos = np.sum(y_train == 1)
        train_neg = np.sum(y_train == 0)
        test_pos = np.sum(y_test == 1)
        test_neg = np.sum(y_test == 0)
        
        print(f"  Training set class distribution: {train_pos} positive, {train_neg} negative")
        print(f"  Testing set class distribution: {test_pos} positive, {test_neg} negative")
    
    # 8. Convert to PyTorch tensors and move to device
    train_inputs = torch.tensor(X_train, dtype=torch.float32).to(device)
    train_labels = torch.tensor(y_train, dtype=torch.long).to(device)
    test_inputs = torch.tensor(X_test, dtype=torch.float32).to(device)
    test_labels = torch.tensor(y_test, dtype=torch.long).to(device)
    
    # 9. Create dataset dictionary
    dataset = {
        'train_input': train_inputs,
        'train_label': train_labels,
        'test_input': test_inputs,
        'test_label': test_labels,
        'label_id': target_label_id
    }
    
    return dataset

# Add validation data loading function
def create_validation_dataset(target_label_id, config, verbose=True):
    """
    Create a validation dataset for a specific label using actual validation data
    
    Args:
        target_label_id: ID of the target label (positive class)
        config: Configuration object with hyperparameters
        verbose: Whether to print progress information
        
    Returns:
        validation_dataset: Dictionary containing validation data
    """
    if verbose:
        print(f"Creating validation dataset for label {target_label_id}...")
        
    # Load validation data for the target label
    val_file = get_label_file_path(target_label_id, is_validation=True)
    if not val_file:
        if verbose:
            print(f"  No validation data found for label {target_label_id}, will use test split from training data")
        return None
        
    # Load positive samples from validation set
    val_positive_samples = np.load(val_file)
    val_num_positive = len(val_positive_samples)
    
    if verbose:
        print(f"  Loaded {val_num_positive} positive validation samples")
    
    # Calculate how many negative validation samples we need
    val_num_negative = val_num_positive * config.val_negative_ratio
    
    # Load validation label index file
    val_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_set_by_label'
    val_index_file = os.path.join(val_label_dir, "val_label_index.txt")
    
    if not os.path.exists(val_index_file):
        if verbose:
            print(f"  Validation index file not found: {val_index_file}")
        return None
        
    val_label_info = load_label_index(val_index_file)
    val_valid_labels = [label_id for label_id, info in val_label_info.items() 
                       if info['count'] > 0 and label_id != target_label_id]
    
    # Get negative samples from other validation labels
    val_negative_samples = []
    val_negative_count = 0
    
    # Use all other labels as negative samples
    for other_label_id in val_valid_labels:
        other_val_file = get_label_file_path(other_label_id, is_validation=True)
        if not other_val_file:
            continue
            
        try:
            # Load validation data for this label
            label_samples = np.load(other_val_file)
            
            # Calculate how many samples to take proportionally
            label_size = val_label_info[other_label_id]['count']
            total_other_size = sum(val_label_info[l]['count'] for l in val_valid_labels)
            proportion = label_size / total_other_size if total_other_size > 0 else 0
            
            samples_needed = min(len(label_samples), max(5, int(val_num_negative * proportion)))
            
            # Randomly sample if we have more than needed
            if samples_needed < len(label_samples):
                indices = np.random.choice(len(label_samples), samples_needed, replace=False)
                sampled = label_samples[indices]
            else:
                sampled = label_samples
                
            val_negative_samples.append(sampled)
            val_negative_count += len(sampled)
            
            # Check if we have enough negative samples
            if val_negative_count >= val_num_negative:
                break
                
        except Exception as e:
            if verbose:
                print(f"  Error loading validation data for label {other_label_id}: {str(e)}")
    
    # Combine negative validation samples
    if val_negative_samples:
        val_negative_samples = np.vstack(val_negative_samples)
        
        # If we have more than needed, randomly sample
        if len(val_negative_samples) > val_num_negative:
            indices = np.random.choice(len(val_negative_samples), int(val_num_negative), replace=False)
            val_negative_samples = val_negative_samples[indices]
    else:
        if verbose:
            print("  No negative validation samples could be loaded")
        return None
        
    if verbose:
        print(f"  Loaded {len(val_negative_samples)} negative validation samples")
        
    # Create validation features and labels
    X_val = np.vstack([val_positive_samples, val_negative_samples])
    y_val_positive = np.ones(val_num_positive, dtype=np.int64)
    y_val_negative = np.zeros(len(val_negative_samples), dtype=np.int64)
    y_val = np.concatenate([y_val_positive, y_val_negative])
    
    # Shuffle validation data
    val_indices = np.arange(X_val.shape[0])
    np.random.shuffle(val_indices)
    X_val = X_val[val_indices]
    y_val = y_val[val_indices]
    
    # Normalize validation features if needed
    if config.normalize_features:
        scaler = StandardScaler()
        X_val = scaler.fit_transform(X_val)
        
    # Convert to PyTorch tensors
    val_inputs = torch.tensor(X_val, dtype=torch.float32).to(device)
    val_labels = torch.tensor(y_val, dtype=torch.long).to(device)
    
    # Create validation dataset dictionary
    validation_dataset = {
        'input': val_inputs,
        'label': val_labels,
        'label_id': target_label_id
    }
    
    if verbose:
        print(f"  Validation dataset created with {len(val_inputs)} samples")
        print(f"  Positive samples: {val_num_positive}, Negative samples: {len(val_negative_samples)}")
        
    return validation_dataset

# Load the dataset for a specific label
target_label = 15  # Change this to the label you want to classify

# Try to use real data first
try:
    dataset = create_1vs_rest_dataset(target_label, config)
    validation_data = create_validation_dataset(target_label, config)
    
    # If we have separate validation data, add it to the dataset
    if validation_data is not None:
        dataset['val_input'] = validation_data['input']
        dataset['val_label'] = validation_data['label']
    
    print("Successfully loaded real brain voxel data")
except Exception as e:
    print(f"Could not load real data: {e}")
    print("Creating synthetic demo dataset instead")
    
    # Fallback to synthetic data if real data loading fails
    def create_demo_dataset(target_label_id, config, verbose=True):
        """Create a simulated dataset for demonstration purposes"""
        if verbose:
            print(f"Creating simulated dataset for label {target_label_id}...")
        
        # Set dimensions
        num_positive = 500  # Simulate 500 positive samples
        num_negative = num_positive * config.negative_ratio
        
        # Create synthetic features (random for demonstration)
        positive_features = np.random.randn(num_positive, config.feature_dim)
        # Make negative samples slightly different distribution
        negative_features = np.random.randn(int(num_negative), config.feature_dim) * 1.2 + 0.5
        
        # Combine features and create labels
        X = np.vstack([positive_features, negative_features])
        y_positive = np.ones(num_positive, dtype=np.int64)
        y_negative = np.zeros(int(num_negative), dtype=np.int64)
        y = np.concatenate([y_positive, y_negative])
        
        # Shuffle the data
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]
        
        # Split into training and testing sets
        test_size = int(X.shape[0] * config.test_size)
        X_train, X_test = X[:-test_size], X[-test_size:]
        y_train, y_test = y[:-test_size], y[-test_size:]
        
        if verbose:
            print(f"  Created simulated dataset with {X.shape[0]} samples")
            print(f"  Training set: {X_train.shape[0]} samples")
            print(f"  Testing set: {X_test.shape[0]} samples")
        
        # Convert to PyTorch tensors and move to device
        train_inputs = torch.tensor(X_train, dtype=torch.float32).to(device)
        train_labels = torch.tensor(y_train, dtype=torch.long).to(device)
        test_inputs = torch.tensor(X_test, dtype=torch.float32).to(device)
        test_labels = torch.tensor(y_test, dtype=torch.long).to(device)
        
        # Create dataset dictionary
        dataset = {
            'train_input': train_inputs,
            'train_label': train_labels,
            'test_input': test_inputs,
            'test_label': test_labels,
            'label_id': target_label_id
        }
        
        return dataset
    
    dataset = create_demo_dataset(target_label, config)

# Print dataset summary
print(f"Train data shape: {dataset['train_input'].shape}")
print(f"Train label shape: {dataset['train_label'].shape}")
print(f"Test data shape: {dataset['test_input'].shape}")
print(f"Test label shape: {dataset['test_label'].shape}")
print("====================================")

Creating 1-vs-rest dataset for label 15...
  Found 101 valid labels with training data
  Loaded 9441 positive samples from /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/train_set_by_label/label_15_count_9441_voxels.npy
  Added 312 negative samples from label 59
  Added 171 negative samples from label 27
  Added 957 negative samples from label 5
  Added 394 negative samples from label 13
  Added 759 negative samples from label 63
  Final negative sample count: 94371 from 100 labels
  Total dataset size: 103812 samples
  Training set: 83050 samples
  Testing set: 20762 samples
  Feature dimension: 341
  Training set class distribution: 7584 positive, 75466 negative
  Testing set class distribution: 1857 positive, 18905 negative
Creating validation dataset for label 15...
  Loaded 272 positive validation samples
  Loaded 1360 negative validation samples
  Validation dataset created with 1632 samples
  Positive samples: 272, Negative samples: 1360
Successfully

In [6]:
# Cell 4 (Simplified): KAN Model Initialization

import torch
from kan import *
import matplotlib.pyplot as plt

# 使用与示例相同的简单初始化方式
model = KAN(width=[341, 256, 128, 2], grid=10, k=3, seed=0, device=device)

# 执行前向传播来初始化模型
_ = model(dataset['train_input'][:10])

print(f"KAN模型已创建: 输入维度={341}, 输出维度=2")
print(f"网格大小: 10, k值: 3")
print(f"总参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# # 尝试绘制模型结构
# try:
#     model.plot(beta=100, scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
# except:
#     print("无法绘制模型结构图，但这不影响训练")

checkpoint directory created: ./model
saving model version 0.0
KAN模型已创建: 输入维度=341, 输出维度=2
网格大小: 10, k值: 3
总参数数量: 2286080


In [ ]:
# Cell 5 (Simplified): Training Functions

# 定义准确率度量函数
def train_acc():
    return torch.mean((torch.argmax(model(dataset['train_input']), dim=1) == dataset['train_label']).float())

def test_acc():
    return torch.mean((torch.argmax(model(dataset['test_input']), dim=1) == dataset['test_label']).float())

# 与示例相同的训练方法
print("开始训练模型...")
results = model.fit(dataset, opt="Adam", metrics=(train_acc, test_acc),
                    loss_fn=torch.nn.CrossEntropyLoss(), steps=100, 
                    lamb=0.01, lamb_entropy=10., save_fig=False, img_folder='video_img')

print("训练完成!")
print(f"最终训练准确率: {results['train_acc'][-1]:.4f}")
print(f"最终测试准确率: {results['test_acc'][-1]:.4f}")

In [ ]:
# Cell 6 (Simplified): Model Pruning and Symbolic Expression

# 模型剪枝
print("对模型进行剪枝...")
model = model.prune()
print("剪枝完成")

# 微调剪枝后的模型
print("微调剪枝后的模型...")
results_1 = model.fit(dataset, opt="Adam", metrics=(train_acc, test_acc),
                     loss_fn=torch.nn.CrossEntropyLoss(), steps=50, 
                     lamb=0.01, lamb_entropy=10.)

print(f"微调后训练准确率: {results_1['train_acc'][-1]:.4f}")
print(f"微调后测试准确率: {results_1['test_acc'][-1]:.4f}")

# 尝试绘制剪枝后的模型
try:
    model.plot(scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
except:
    print("无法绘制剪枝后的模型结构图")

# 提取符号表达式
print("尝试提取符号表达式...")
try:
    lib = ['x','x^2','x^3','exp','log','sqrt','tanh','sin','abs']
    model.auto_symbolic(lib=lib)
    
    # 获取符号公式
    formula1, formula2 = model.symbolic_formula()[0]
    
    print("\n负类符号表达式:")
    print(formula1)
    
    print("\n正类符号表达式:")
    print(formula2)
    
    # 尝试简化公式
    try:
        from sympy import simplify
        print("\n简化后的正类表达式:")
        print(simplify(formula2))
    except:
        print("无法简化公式")
except Exception as e:
    print(f"提取符号表达式失败: {e}")

In [ ]:
# Cell 7 (Simplified): Neural Network Comparison

# 定义一个标准神经网络进行比较
class BrainNet(nn.Module):
    def __init__(self):
        super(BrainNet, self).__init__()
        self.fc1 = nn.Linear(341, 256)  # 341输入到256隐藏节点
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)  # 256到128隐藏节点
        self.fc3 = nn.Linear(128, 2)    # 128到2输出节点

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

# 训练神经网络模型
def train_model(model, train_loader, criterion, optimizer, num_epochs=100):
    model.train()
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        if (epoch+1) % 10 == 0:
            # 每10个epoch输出一次损失
            print(f'神经网络训练: Epoch {epoch+1}, Loss: {loss.item():.4f}')

# 评估神经网络模型
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'神经网络测试准确率: {accuracy:.2f}%')
    return accuracy

# 创建数据加载器
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(dataset['train_input'], dataset['train_label']), 
    batch_size=32, 
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(dataset['test_input'], dataset['test_label']), 
    batch_size=32, 
    shuffle=False
)

# 初始化神经网络
nn_model = BrainNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.01)

# 训练神经网络
print("开始训练神经网络进行比较...")
train_model(nn_model, train_loader, criterion, optimizer)

# 评估神经网络
nn_accuracy = test_model(nn_model, test_loader)

# 比较KAN和神经网络
print("\n模型比较:")
print(f"KAN测试准确率: {results_1['test_acc'][-1]*100:.2f}%")
print(f"神经网络测试准确率: {nn_accuracy:.2f}%")
print(f"差异: {(results_1['test_acc'][-1]*100 - nn_accuracy):.2f}%")

# 打印KAN的优势
print("\nKAN的优势:")
print("1. 可解释性 - 能够提取数学公式解释决策")
print("2. 模型剪枝 - 可以减少模型大小并保持性能")
print("3. 固定网格大小避免训练震荡")

In [ ]:
# Cell 8 (Simplified): Create Video from Training Images

import os
import numpy as np

try:
    import moviepy.video.io.ImageSequenceClip
    
    # 创建视频
    video_name = 'video'
    fps = 10
    
    # 获取图像文件路径
    image_folder = 'video_img'
    files = os.listdir(image_folder)
    train_index = []
    
    # 获取所有数字命名的jpg文件
    for file in files:
        if file[0].isdigit() and file.endswith('.jpg'):
            train_index.append(int(file[:-4]))
    
    # 按正确顺序排序索引
    train_index = np.sort(train_index)
    
    # 创建图像文件路径列表
    image_files = [f'{image_folder}/{idx}.jpg' for idx in train_index]
    
    if image_files:
        # 创建视频并保存
        clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
        clip.write_videofile(f'{video_name}.mp4')
        
        print(f"已创建训练可视化视频: '{video_name}.mp4'")
        print(f"视频包含 {len(image_files)} 帧，帧率为 {fps} fps")
    else:
        print("未找到训练可视化图像文件")
        
except ImportError:
    print("未安装moviepy，跳过视频创建")
except Exception as e:
    print(f"创建训练视频时出错: {e}")